# Bovine Liver Lesion Classification — Kumasi Abattoir Pilot

Analysis notebook for the smartphone liver-lesion study. The heavy lifting lives in
`src/liver/`; this notebook orchestrates it and displays the results.

**Run order matters.** Sections 2–4 change what the numbers *mean*, so running the models
first produces figures that have to be withdrawn.

| Section | What it does |
|---|---|
| 1 | Setup |
| 2 | Manifest — dedupe, assign liver identities |
| 3 | Harmonise images, run the leakage gates |
| 4 | Confound diagnostics — is the signal in the lesion? |
| 5 | Classical baseline (colour + GLCM, SVM / RF / logistic) |
| 6 | Deep learning — three backbones, frozen and fine-tuned |
| 7 | Grad-CAM and occlusion — Objective 5 as a test that can fail |
| 8 | Descriptive statistics for the write-up |

> **Current status.** The section 4 gates **fail** for whole-image classification: an 8×8
> thumbnail scores AUC 0.989 and the image border alone scores 0.971, because normal and
> lesioned livers were photographed under different conditions. Treat every number in
> sections 5–7 as a measure of capture confound, not pathology.
>
> **Section 7b is the usable result.** Segmenting the organ and classifying tissue patches
> gives AUC 0.940 (CI 0.896–0.973) per animal, against a worst leakage probe of 0.715.
> That analysis is defensible on the images already collected.

## 1. Setup

Run in the `analysis` conda environment (torch, scikit-learn, scikit-image).

In [ ]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from liver import (config, manifest, harmonise, probes, features,
                   splits, models, train, evaluate, gradcam)

config.ensure_dirs()
pd.set_option("display.width", 160)
print("project:", config.PROJECT_ROOT)
print("data   :", config.DATA_ROOT)

## 2. Manifest — deduplicate and assign liver identities

The manifest is the single source of truth: path, binary label, lesion type, liver id, md5
and the capture properties needed for auditing.

Two things this fixes. Nine sets of files were **byte-identical** — one normal image was
stored five times. And repeat views of one animal (`LF13a/b/c`, `ABS 11 a/11b`) must stay
in the same fold, or the model is tested on livers it trained on.

In [ ]:
df, dropped = manifest.build(config.DATA_ROOT)
df.to_csv(config.MANIFEST_CSV, index=False)
dropped.to_csv(config.DROPPED_CSV, index=False)

print(f"unique images   : {len(df)}   (dropped {len(dropped)} exact duplicates)")
print(f"distinct livers : {df.liver_id.nunique()}")
df.groupby(df.label.map({0: "normal", 1: "abnormal"})).size()

In [ ]:
# Images per lesion type, AND per distinct animal -- the second column is the one that
# governs whether a class can be modelled at all.
df.groupby("lesion_type").agg(images=("path", "size"),
                              livers=("liver_id", "nunique")
                             ).sort_values("images", ascending=False)

In [ ]:
# Livers contributing more than one image. Left unhandled these leak across folds.
sizes = df.groupby("liver_id").size()
multi = sizes[sizes > 1].sort_values(ascending=False)
print(f"{len(multi)} livers cover {multi.sum()} images\n")
for lid in multi.index[:10]:
    print(f"  {lid}: " + ", ".join(Path(p).name for p in df.loc[df.liver_id == lid, "path"]))

## 3. Harmonise, then check the gates

Every image is decoded, EXIF-rotated, centre-cropped to square and written losslessly as
PNG. That removes the file-level confound: resolution, JPEG quantization tables and EXIF
tags predicted the class at **AUC 1.000** before this step.

Re-encoding as JPEG was tried and rejected — a second compression interacts with the first
and made the artefact *more* class-predictive (0.61 → 0.98).

In [ ]:
df = harmonise.build_harmonised_set(df, config.DATA_ROOT, force=False)
df.to_csv(config.MANIFEST_CSV, index=False)

raw_paths  = [config.DATA_ROOT / p for p in df.path]
harm_paths = [config.PROJECT_ROOT / p for p in df.harmonised_path]
y, groups = df.label.to_numpy(), df.liver_id.to_numpy()

rows = []
for stage, paths in (("RAW", raw_paths), ("HARMONISED", harm_paths)):
    for name, extract, gated in (
        ("capture metadata", probes.capture_metadata_features, True),
        ("compression artefact", probes.compression_artefact_features, True),
        ("image content stats", probes.content_statistics_features, False),
    ):
        rows.append(probes.run_probe(f"[{stage}] {name}", extract(paths), y, groups, gated))

probes.report(rows)

`capture metadata` should fall to 0.500 — it does. `image content stats` is *not* gated:
contrast genuinely differs between a uniform liver and one with white abscess foci, so
skill there is not automatically leakage. Section 4 decides which it is here.

## 4. Confound diagnostics — is the signal in the lesion?

Each variant destroys lesion detail a different way. Anything still scoring high is being
carried by something other than the lesion. **This is the section that decides whether the
dataset can support the study's claim.**

In [ ]:
%run scripts/04_confound_diagnostics.py

**How to read it.** If the border ring or the 8×8 thumbnail scores high, the classes differ
in how they were *photographed*, and no model can separate that from pathology. Targets for
a clean dataset: border ring < 0.65, thumbnail < 0.70.

## 5. Classical baseline

Colour histograms (RGB + HSV) and GLCM texture, with SVM, Random Forest and logistic
regression — through the *same* nested group folds the deep models use, which is what makes
the comparison in §2.6 fair.

In [ ]:
folds = splits.nested_folds(y, groups, n_splits=5, n_repeats=3, seed=config.SEED)
print(splits.describe(folds[:5], y, groups))   # 'overlap' must read 'none' on every row

In [ ]:
%run scripts/03_classical_baseline.py

## 6. Deep learning — three backbones, two regimes

Frozen configurations reuse precomputed embeddings and run in about a minute each.
Fine-tuning is real training and is slow on CPU; add `--finetune` when the data justifies it.

In [ ]:
%run scripts/05_deep_learning.py

In [ ]:
# Add --finetune for the full 3 x 2 grid (hours on a 4-thread CPU).
# %run scripts/05_deep_learning.py --finetune

## 7. Grad-CAM and occlusion — Objective 5

Objective 5 asks whether predictions are driven by the lesion. That can be false, so it is
measured rather than illustrated:

- **Border attention share** — how much activation lands in the outer 20% frame. A uniform
  map gives 0.64; higher means the model is looking outside the organ.
- **Occlusion** — slide a patch across the image and record how far the prediction moves.
  If hiding a region changes nothing, that region was not being used.

In [ ]:
%run scripts/06_gradcam_occlusion.py

In [ ]:
from IPython.display import Image as IPImage
IPImage(filename=str(config.OUTPUTS / "figures" / "gradcam_occlusion.png"), width=760)

## 8. Descriptive statistics for the write-up

Objective 6 — lesion frequencies for Chapter Three, reported by distinct animal as well as
by image.

In [ ]:
desc = df.groupby("lesion_type").agg(images=("path", "size"),
                                     livers=("liver_id", "nunique"))
desc["images_pct"] = (100 * desc.images / desc.images.sum()).round(1)
desc = desc.sort_values("images", ascending=False)
desc.to_csv(config.OUTPUTS / "lesion_frequency.csv")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
desc.images.plot(kind="barh", ax=axes[0], color="#8C2F39")
axes[0].set_title("Images per lesion type"); axes[0].set_xlabel("images")
desc.livers.plot(kind="barh", ax=axes[1], color="#5B4B8A")
axes[1].set_title("Distinct animals per lesion type"); axes[1].set_xlabel("animals")
plt.tight_layout(); plt.savefig(config.OUTPUTS / "figures" / "lesion_frequency.png", dpi=150)
plt.show()
desc

## 7b. Tissue-only patch analysis — the usable result

Sections 4–7 establish that whole-image classification on this dataset measures capture
conditions. This section works around that instead of ignoring it.

The confound lives in the **background** and the **framing**, so both are discarded: the
organ is segmented, patches are sampled from well inside it, and what gets classified is
tissue rather than a photograph. Patch size scales with the organ's apparent area, so a
close-up and a whole-organ shot contribute tissue at comparable magnification. Texture runs
on lightness standardised within each patch, making it invariant to session lighting.

Each confound gets its own probe — colour, compression, magnification — and the task has to
beat all of them.

In [ ]:
%run scripts/07_patch_analysis.py

**Result.** Texture only, scored per animal over 132 animals: **AUC 0.940 (95% CI
0.896–0.973)**, against a worst leakage probe of 0.715. A margin of 0.225 that cannot be
explained by background, framing, lighting or zoom.

**Report the texture number, not the colour one.** The mean-colour probe sits at 0.715, so
the colour pathway is still partly carrying session differences. Texture is the pathway
with a clean provenance.

Caveats for the write-up: segmentation is classical and imperfect, so a minority of patches
may include non-parenchyma; and this is still one site, one collection period, 132 animals.

In [ ]:
# Visual check on the segmentation the result depends on -- always look at this before
# quoting the number.
import numpy as np
from PIL import Image
from liver import segment

sample = pd.concat([df[df.label == 0].sample(4, random_state=3),
                    df[df.label == 1].sample(4, random_state=3)])
fig, axes = plt.subplots(3, 8, figsize=(16, 6))
for j, (_, r) in enumerate(sample.iterrows()):
    im = Image.open(config.PROJECT_ROOT / r.harmonised_path)
    mask = segment.liver_mask(im, seed=config.SEED)
    arr = np.asarray(im.convert("RGB"))
    axes[0, j].imshow(arr)
    axes[0, j].set_title(("normal" if r.label == 0 else r.lesion_type)[:12], fontsize=8)
    dimmed = arr.copy(); dimmed[~mask] = (dimmed[~mask] * 0.18).astype(np.uint8)
    axes[1, j].imshow(dimmed); axes[1, j].set_title(f"mask {mask.mean():.0%}", fontsize=8)
    ps = segment.organ_relative_patches(im, mask, n_patches=3, seed=config.SEED)
    if ps:
        axes[2, j].imshow(np.hstack(ps[:3]))
    axes[2, j].set_title("tissue patches", fontsize=8)
    for i in range(3):
        axes[i, j].axis("off")
plt.tight_layout()
plt.savefig(config.OUTPUTS / "figures" / "segmentation_check.png", dpi=110)
plt.show()

## Next steps

**Now, with the images already collected.** Section 7b is the result to write up: tissue
texture, AUC 0.940 per animal, with every confound probe reported alongside it. That is a
publishable pilot. Objective 6 (lesion frequencies, section 8) also needs no new data.

**As new collection proceeds.**

1. **Photograph both classes in the same sessions**, same surface, same distance and
   framing, same device, files transferred directly off the phone. This is what unblocks
   whole-image classification — no preprocessing fixed it.
2. **Record an animal identifier for every image**, so liver grouping never has to be
   reconstructed from image similarity again.
3. **Re-run sections 2–4 as the acceptance test** on each new batch. If the gates do not
   pass, more images will not help.
4. **Only then trust sections 5–7**, and require the border-attention number to sit below
   the 0.64 uniform reference before quoting any performance figure.
5. **Have the inspector score Grad-CAM maps blind** — does the highlighted region contain
   the lesion? That percentage is the real Objective 5 result.

Proposal amendments: §2.3 sample counts, §2.5 categories (add telangiectasis, drop fatty
liver), §2.4 collection protocol, §2.10 limitations. Sample-size targets should be counted
in distinct animals, not images.
